# Convolutional Neural Network

### Importing the libraries

# Convolutional Neural Network

In [1]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.preprocessing.image import ImageDataGenerator

tf.get_logger().setLevel('ERROR')

gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('GPU tidak terdeteksi. Restart kernel, pilih Python (.venv cnn), lalu jalankan ulang dari cell pertama.')

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

mixed_precision.set_global_policy('mixed_float16')


In [2]:
tf.__version__

'2.21.0'

In [3]:
# Ubah parameter di sini untuk eksperimen GPU
# Variasi 4 laporan: filter awal 64 dan kernel 5x5.
IMAGE_SIZE = (180, 180)
BATCH_SIZE = 64
VALIDATION_SPLIT = 0.2
EPOCHS = 100
FILTERS = 64        # Variasi 4
KERNEL_SIZE = 5     # Kernel 5x5
L2_REG = 0.00001
VARIANT_ID = f'f{FILTERS}_k{KERNEL_SIZE}_img{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}'
RESULTS_DIR = 'results'
PRED_IMAGE_PATHS = ['single_prediction/cat_or_dog_1.jpg', 'single_prediction/cat_or_dog_2.jpg']
RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.1,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=(0.85, 1.15),
    horizontal_flip=True,
    rotation_range=20,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT
)
training_set = train_datagen.flow_from_directory('training_set',
                                                 target_size=IMAGE_SIZE,
                                                 batch_size=BATCH_SIZE,
                                                 class_mode='binary',
                                                 subset='training',
                                                 seed=RANDOM_SEED)

validation_datagen = ImageDataGenerator(rescale=1./255, validation_split=VALIDATION_SPLIT)
validation_set = validation_datagen.flow_from_directory('training_set',
                                                   target_size=IMAGE_SIZE,
                                                   batch_size=BATCH_SIZE,
                                                   class_mode='binary',
                                                   subset='validation',
                                                   shuffle=False,
                                                   seed=RANDOM_SEED)

Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.


### Preprocessing the Test set

In [5]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_set = test_datagen.flow_from_directory('test_set',
                                            target_size=IMAGE_SIZE,
                                            batch_size=BATCH_SIZE,
                                            class_mode='binary',
                                            shuffle=False)

Found 2000 images belonging to 2 classes.


### Initialising the CNN

In [6]:
cnn = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=[IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
])

### Step 1 - Convolution

In [7]:
# Block 1: double conv (VGG-style) — 2 conv sebelum pooling, fitur lebih kaya
cnn.add(tf.keras.layers.Conv2D(
    filters=FILTERS, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)
))
cnn.add(tf.keras.layers.Conv2D(
    filters=FILTERS, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)
))
cnn.add(tf.keras.layers.BatchNormalization())

### Step 2 - Pooling

In [8]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [9]:
# Block 2: FILTERS*2, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 2, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 2, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Block 3: FILTERS*4, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Block 4: FILTERS*4, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Block 5: fitur lebih dalam, tapi feature map sudah kecil sehingga masih ramah GPU
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 8, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 8, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [10]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [11]:
cnn.add(tf.keras.layers.Dense(units=256, activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Dropout(0.45))
cnn.add(tf.keras.layers.Dense(units=128, activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Dropout(0.25))

### Step 5 - Output Layer

In [12]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid', dtype='float32'))
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 180, 180, 64)   │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 180, 180, 64)   │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 180, 180, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 90, 90, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 90, 90, 128)    │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 90, 90, 128)    │       409,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 90, 90, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 45, 45, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 45, 45, 256)    │       819,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 45, 45, 256)    │     1,638,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 45, 45, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 22, 22, 256)    │     1,638,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 22, 22, 256)    │     1,638,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 22, 22, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 11, 11, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 11, 11, 512)    │     3,277,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 11, 11, 512)    │     6,554,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 11, 11, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 5, 5, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     3,277,056 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 19,603,777 (74.78 MB)

 Trainable params: 19,601,345 (74.77 MB)

 Non-trainable params: 2,432 (9.50 KB)

## Part 3 - Training the CNN

### Compiling the CNN

In [13]:
# Learning rate dibuat lebih halus untuk training lebih panjang.
optimizer = tf.keras.optimizers.Adam(learning_rate=0.00025)
cnn.compile(optimizer = optimizer, loss = 'binary_crossentropy', metrics = ['accuracy'])

In [14]:
import json
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

run_started_at = datetime.now(ZoneInfo('Asia/Jakarta'))
run_timestamp = run_started_at.strftime('%Y-%m-%d %H:%M:%S WIB')
run_stamp = run_started_at.strftime('%Y%m%d_%H%M%S')
base_run_dir = Path(RESULTS_DIR) / f'{VARIANT_ID}_{run_stamp}'
run_dir = base_run_dir
duplicate_index = 1
while run_dir.exists():
    run_dir = Path(f'{base_run_dir}_{duplicate_index:02d}')
    duplicate_index += 1
run_dir.mkdir(parents=True, exist_ok=False)

RUN_ID = run_dir.name
CHECKPOINT_PATH = str(run_dir / f'best_model_{RUN_ID}.keras')
RUN_METRICS_PATH = str(run_dir / f'metrics_{RUN_ID}.json')
RUN_REPORT_PATH = str(run_dir / f'training_report_{RUN_ID}.txt')

print('\n=== IDENTITAS RUN ===')
print(f'Waktu run dimulai: {run_timestamp}')
print(f'Variant ID: {VARIANT_ID}')
print(f'FILTERS: {FILTERS}')
print(f'KERNEL_SIZE: {KERNEL_SIZE}')
print(f'IMAGE_SIZE: {IMAGE_SIZE}')
print(f'Run folder: {run_dir}')
print(f'Model checkpoint baru: {CHECKPOINT_PATH}')
print(f'Metrics JSON baru: {RUN_METRICS_PATH}')
print(f'Report TXT baru: {RUN_REPORT_PATH}')

early_stop = EarlyStopping(monitor='val_accuracy', mode='max', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', mode='max', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
checkpoint = ModelCheckpoint(CHECKPOINT_PATH, monitor='val_accuracy', mode='max', save_best_only=True, verbose=1)

history = cnn.fit(x=training_set, validation_data=validation_set, epochs=EPOCHS, callbacks=[early_stop, reduce_lr, checkpoint], verbose=1)

history_data = history.history
epochs_ran = len(history_data['accuracy'])
best_val_epoch = history_data['val_accuracy'].index(max(history_data['val_accuracy'])) + 1
best_train_epoch = history_data['accuracy'].index(max(history_data['accuracy'])) + 1
best_val_index = best_val_epoch - 1
final_epoch_index = epochs_ran - 1

train_loss_at_best_val = history_data['loss'][best_val_index]
train_acc_at_best_val = history_data['accuracy'][best_val_index]
val_loss_at_best_val = history_data['val_loss'][best_val_index]
val_acc_at_best_val = history_data['val_accuracy'][best_val_index]
best_train_acc = max(history_data['accuracy'])
final_train_loss = history_data['loss'][final_epoch_index]
final_train_acc = history_data['accuracy'][final_epoch_index]
final_val_loss = history_data['val_loss'][final_epoch_index]
final_val_acc = history_data['val_accuracy'][final_epoch_index]

print('\n=== RINGKASAN TRAINING ===')
print(f'Tanggal output dicetak: {run_timestamp}')
print(f'Epoch dijalankan: {epochs_ran}/{EPOCHS}')
print(f'Best val epoch: {best_val_epoch}')
print(f'Train loss at best val epoch: {train_loss_at_best_val:.4f}')
print(f'Train acc at best val epoch: {train_acc_at_best_val * 100:.2f}%')
print(f'Val loss at best val epoch: {val_loss_at_best_val:.4f}')
print(f'Val acc at best val epoch: {val_acc_at_best_val * 100:.2f}%')
print(f'Best train acc: {best_train_acc * 100:.2f}% (epoch {best_train_epoch})')
print(f'Final epoch train loss: {final_train_loss:.4f}')
print(f'Final epoch train acc: {final_train_acc * 100:.2f}%')
print(f'Final epoch val loss: {final_val_loss:.4f}')
print(f'Final epoch val acc: {final_val_acc * 100:.2f}%')
print(f'Best model saved to: {CHECKPOINT_PATH}')

print('\n=== DATASET YANG DIPAKAI ===')
print(f'Training images: {training_set.samples}')
print(f'Validation images: {validation_set.samples}')
print(f'Test images: {test_set.samples}')
print(f'Class indices: {training_set.class_indices}')

history_metric_keys = list(history_data.keys())
print('\n=== RIWAYAT METRIK TRAINING PER EPOCH ===')
print('Metrics tersimpan:', history_metric_keys)
print('epoch | ' + ' | '.join(history_metric_keys))
for epoch_index in range(epochs_ran):
    values = []
    for metric_name in history_metric_keys:
        metric_value = float(history_data[metric_name][epoch_index])
        if metric_name in ('learning_rate', 'lr'):
            formatted_value = f'{metric_value:.6g}'
        elif 'accuracy' in metric_name or metric_name in ('auc', 'precision', 'recall'):
            formatted_value = f'{metric_value * 100:.2f}%'
        else:
            formatted_value = f'{metric_value:.4f}'
        values.append(formatted_value)
    print(f'{epoch_index + 1:>5} | ' + ' | '.join(values))

test_set.reset()
test_loss, test_accuracy = cnn.evaluate(test_set, verbose=0)

print('\n=== EVALUASI TEST SET ===')
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy total: {test_accuracy * 100:.2f}%')

test_set.reset()
test_probabilities = cnn.predict(test_set, verbose=0).reshape(-1)
test_predictions = (test_probabilities >= 0.5).astype('int32')
class_indices = training_set.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}
confusion_matrix = tf.math.confusion_matrix(
    test_set.classes,
    test_predictions,
    num_classes=len(training_set.class_indices)
).numpy()

print('class_indices:', class_indices)
print('Confusion matrix rows=true, cols=pred:')
print(confusion_matrix)

display_names = {'cats': 'kucing', 'dogs': 'anjing'}

def print_class_result(class_label, title):
    true_index = class_indices[class_label]
    display_label = display_names.get(class_label, class_label)
    total = int(confusion_matrix[true_index].sum())
    correct = int(confusion_matrix[true_index, true_index])
    predicted_total = int(confusion_matrix[:, true_index].sum())
    false_negative = total - correct
    false_positive = predicted_total - correct
    recall = (correct / total) if total else 0.0
    precision = (correct / predicted_total) if predicted_total else 0.0
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    print(f'\n=== {title} ===')
    print(f'Label model: {class_label} (index {true_index})')
    print(f'Total gambar {display_label}: {total}')
    print(f'Total diprediksi sebagai {display_label}: {predicted_total}')
    print(f'Benar sebagai {display_label}: {correct}')
    for pred_index in range(len(idx_to_class)):
        if pred_index != true_index:
            pred_label = idx_to_class[pred_index]
            pred_display = display_names.get(pred_label, pred_label)
            wrong_count = int(confusion_matrix[true_index, pred_index])
            print(f'Salah jadi {pred_display}: {wrong_count}')
    print(f'False negative {display_label}: {false_negative}')
    print(f'False positive {display_label}: {false_positive}')
    print(f'Precision {display_label}: {precision * 100:.2f}%')
    print(f'Recall/accuracy {display_label}: {recall * 100:.2f}%')
    print(f'F1-score {display_label}: {f1_score * 100:.2f}%')

    return {
        'label': class_label,
        'display_label': display_label,
        'index': int(true_index),
        'total': total,
        'predicted_total': predicted_total,
        'correct': correct,
        'false_negative': false_negative,
        'false_positive': false_positive,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
    }

cats_result = print_class_result('cats', 'HASIL KUCING')
dogs_result = print_class_result('dogs', 'HASIL ANJING')

history_serializable = {
    metric_name: [float(value) for value in values]
    for metric_name, values in history_data.items()
}
metrics_payload = {
    'run_id': RUN_ID,
    'run_timestamp': run_timestamp,
    'variant_id': VARIANT_ID,
    'parameters': {
        'image_size': list(IMAGE_SIZE),
        'batch_size': BATCH_SIZE,
        'validation_split': VALIDATION_SPLIT,
        'epochs_requested': EPOCHS,
        'epochs_ran': epochs_ran,
        'filters': FILTERS,
        'kernel_size': KERNEL_SIZE,
        'l2_reg': L2_REG,
        'random_seed': RANDOM_SEED,
    },
    'paths': {
        'run_dir': str(run_dir),
        'checkpoint': CHECKPOINT_PATH,
        'metrics_json': RUN_METRICS_PATH,
        'report_txt': RUN_REPORT_PATH,
    },
    'summary': {
        'best_val_epoch': best_val_epoch,
        'best_train_epoch': best_train_epoch,
        'train_loss_at_best_val_epoch': float(train_loss_at_best_val),
        'train_accuracy_at_best_val_epoch': float(train_acc_at_best_val),
        'val_loss_at_best_val_epoch': float(val_loss_at_best_val),
        'val_accuracy_at_best_val_epoch': float(val_acc_at_best_val),
        'best_train_accuracy': float(best_train_acc),
        'final_train_loss': float(final_train_loss),
        'final_train_accuracy': float(final_train_acc),
        'final_val_loss': float(final_val_loss),
        'final_val_accuracy': float(final_val_acc),
        'test_loss': float(test_loss),
        'test_accuracy': float(test_accuracy),
    },
    'dataset': {
        'training_images': int(training_set.samples),
        'validation_images': int(validation_set.samples),
        'test_images': int(test_set.samples),
        'class_indices': {key: int(value) for key, value in class_indices.items()},
    },
    'history': history_serializable,
    'confusion_matrix': confusion_matrix.astype(int).tolist(),
    'per_class': {
        'cats': cats_result,
        'dogs': dogs_result,
    },
}

with open(RUN_METRICS_PATH, 'w', encoding='utf-8') as metrics_file:
    json.dump(metrics_payload, metrics_file, indent=2)

report_lines = [
    '=== IDENTITAS RUN ===',
    f'Waktu run dimulai: {run_timestamp}',
    f'Variant ID: {VARIANT_ID}',
    f'FILTERS: {FILTERS}',
    f'KERNEL_SIZE: {KERNEL_SIZE}',
    f'IMAGE_SIZE: {IMAGE_SIZE}',
    f'Run folder: {run_dir}',
    f'Model checkpoint: {CHECKPOINT_PATH}',
    '',
    '=== RINGKASAN TRAINING ===',
    f'Epoch dijalankan: {epochs_ran}/{EPOCHS}',
    f'Best val epoch: {best_val_epoch}',
    f'Train acc at best val epoch: {train_acc_at_best_val * 100:.2f}%',
    f'Val acc at best val epoch: {val_acc_at_best_val * 100:.2f}%',
    f'Best train acc: {best_train_acc * 100:.2f}% (epoch {best_train_epoch})',
    f'Test accuracy total: {test_accuracy * 100:.2f}%',
    '',
    '=== HASIL KUCING ===',
    f"Total gambar kucing: {cats_result['total']}",
    f"Benar sebagai kucing: {cats_result['correct']}",
    f"False positive kucing: {cats_result['false_positive']}",
    f"False negative kucing: {cats_result['false_negative']}",
    f"Recall/accuracy kucing: {cats_result['recall'] * 100:.2f}%",
    f"Precision kucing: {cats_result['precision'] * 100:.2f}%",
    f"F1-score kucing: {cats_result['f1_score'] * 100:.2f}%",
    '',
    '=== HASIL ANJING ===',
    f"Total gambar anjing: {dogs_result['total']}",
    f"Benar sebagai anjing: {dogs_result['correct']}",
    f"False positive anjing: {dogs_result['false_positive']}",
    f"False negative anjing: {dogs_result['false_negative']}",
    f"Recall/accuracy anjing: {dogs_result['recall'] * 100:.2f}%",
    f"Precision anjing: {dogs_result['precision'] * 100:.2f}%",
    f"F1-score anjing: {dogs_result['f1_score'] * 100:.2f}%",
    '',
    '=== FILE OUTPUT ===',
    f'Metrics JSON: {RUN_METRICS_PATH}',
    f'Report TXT: {RUN_REPORT_PATH}',
]
with open(RUN_REPORT_PATH, 'w', encoding='utf-8') as report_file:
    report_file.write('\n'.join(report_lines) + '\n')

print('\n=== FILE OUTPUT RUN ===')
print(f'Model checkpoint: {CHECKPOINT_PATH}')
print(f'Metrics JSON: {RUN_METRICS_PATH}')
print(f'Report TXT: {RUN_REPORT_PATH}')


=== IDENTITAS RUN ===
Waktu run dimulai: 2026-04-27 16:15:25 WIB
Variant ID: f64_k5_img180x180
FILTERS: 64
KERNEL_SIZE: 5
IMAGE_SIZE: (180, 180)
Run folder: results/f64_k5_img180x180_20260427_161525
Model checkpoint baru: results/f64_k5_img180x180_20260427_161525/best_model_f64_k5_img180x180_20260427_161525.keras
Metrics JSON baru: results/f64_k5_img180x180_20260427_161525/metrics_f64_k5_img180x180_20260427_161525.json
Report TXT baru: results/f64_k5_img180x180_20260427_161525/training_report_f64_k5_img180x180_20260427_161525.txt
Epoch 1/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - accuracy: 0.5214 - loss: 1.0536
Epoch 1: val_accuracy improved from None to 0.49625, saving model to results/f64_k5_img180x180_20260427_161525/best_model_f64_k5_img180x180_20260427_161525.keras

Epoch 1: finished saving model to results/f64_k5_img180x180_20260427_161525/best_model_f64_k5_img180x180_20260427_161525.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 123s 393ms/step - accuracy: 0.5295 - loss: 0.8994 - val

## Part 4 - Making a single prediction

In [15]:
import numpy as np
from tensorflow.keras.preprocessing import image

class_indices = training_set.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

print('class_indices:', class_indices)
for pred_path in PRED_IMAGE_PATHS:
    test_image = image.load_img(pred_path, target_size = IMAGE_SIZE)
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis = 0)
    test_image = test_image / 255.0

    result = cnn.predict(test_image, verbose = 0)
    probability = float(result[0][0])
    predicted_index = 1 if probability >= 0.5 else 0
    predicted_label = idx_to_class[predicted_index]

    file_name = pred_path.split('/')[-1]
    print(f"{file_name}: prob={probability:.4f}, pred={predicted_label}")

class_indices: {'cats': 0, 'dogs': 1}
cat_or_dog_1.jpg: prob=1.0000, pred=dogs
cat_or_dog_2.jpg: prob=0.0000, pred=cats


In [16]:
# Dinonaktifkan: output prediksi sekarang sudah dicetak per file di cell sebelumnya
